# Classification

## Setup

In [66]:
from sklearn.model_selection import cross_val_score
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.model_selection import RandomizedSearchCV
import scipy.stats as st
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingRandomSearchCV
from sklearn.ensemble import RandomForestClassifier
import pandas as pd
import numpy as np

## Load Data

In [1]:
from sklearn.datasets import fetch_openml

mnist = fetch_openml('mnist_784', version=1)
X, y = mnist["data"], mnist["target"]

In [52]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=10000, random_state=42
)

## KNeighboursClassifier

In [86]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier

In [76]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=10000, stratify=y, random_state=42
)

In [82]:
train_dist = pd.Series(y_train).value_counts(normalize=True).sort_index() * 100
test_dist  = pd.Series(y_test).value_counts(normalize=True).sort_index() * 100

df = pd.DataFrame({
    "train %": train_dist.round(2),
    "test %":  test_dist.round(2)
})

print(df)

       train %  test %
class                 
0         9.86    9.86
1        11.25   11.25
2         9.98    9.99
3        10.20   10.20
4         9.75    9.75
5         9.02    9.02
6         9.82    9.82
7        10.42   10.42
8         9.75    9.75
9         9.94    9.94


In [ ]:
# speed up: quantization
# Cut memory and speed up math
X_train = X_train.astype(np.int32)   # downcast
X_test = X_test.astype(np.int32)
# - not needed since StandardScaler is used further!!!

In [89]:
pipe = Pipeline([
    ("scaler", StandardScaler()), # TODO: add quantization in pipeline?
    ("pca", PCA(svd_solver="full")),
    ("knn", KNeighborsClassifier(algorithm="brute", n_jobs=1))
])

In [92]:
param_dist = {
    "pca__n_components": st.uniform(0.90, 0.10),   # sample between 0.90–1.00 variance
    "knn__n_neighbors": st.randint(3, 10),        # sample 3–9 neighbors
    "knn__weights": ["uniform", "distance"],
    "knn__p": [2],                                # could add 1 if you want Manhattan too
}

rs = RandomizedSearchCV(
    pipe,
    param_distributions=param_dist,
    n_iter=8,          # try 8–20 depending on time
    cv=2,
    n_jobs=-1,
    scoring="accuracy",
    random_state=42,
    verbose=1
)
rs.fit(X_train, y_train)

Fitting 2 folds for each of 8 candidates, totalling 16 fits


RandomizedSearchCV(cv=2,
                   estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                             ('pca', PCA(svd_solver='full')),
                                             ('knn',
                                              KNeighborsClassifier(algorithm='brute',
                                                                   n_jobs=1))]),
                   n_iter=8, n_jobs=-1,
                   param_distributions={'knn__n_neighbors': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7e31b42d9070>,
                                        'knn__p': [2],
                                        'knn__weights': ['uniform', 'distance'],
                                        'pca__n_components': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x7e31b600c230>},
                   random_state=42, scoring='accuracy', verbose=1)

In [93]:
print("Best params:", rs.best_params_)
print("CV best score:", rs.best_score_)
print("Test acc:", rs.score(X_test, y_test))

Best params: {'knn__n_neighbors': 6, 'knn__p': 2, 'knn__weights': 'distance', 'pca__n_components': np.float64(0.9212339110678276)}
CV best score: 0.9458833333333334
Test acc: 0.9556


TODO: continue trying